In [10]:
### Batch Rating Curve Evaluation Notebook ###

# Imports
import importlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from io import StringIO

import rating_curve_utils as rcu
importlib.reload(rcu)   

# Output and display settings 
OUT_DIR = Path("Plots_Discharge") / "Evaluate_Rating_Curves"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DPI = 300

# Color and palette options
COLORS = {
    "obs_hist": "#1f77b4",    # blue
    "obs_post": "#ff7f0e",    # orange
    "one_to_one": "#990099",  # magenta for 1:1 line
    "rc_line": "#ff00ff"      # RC fit line color
}
PALETTE_MODE = "two-tone"
EVAL_AT = "target_station"

# Date windows you analyze (batch-wide globals)
HIST_START, HIST_END = "2006-01-01", "2014-05-31"
FULL_START, FULL_END = "2006-01-01", "2025-09-01"
SHORT_GAP_DAYS = 3

In [11]:
### Data load and shared objects ###

objs = joblib.load("cache/tributary_cache.joblib")
flow_data_reindexed = objs.get("flow_data_reindexed")
filled_flow_data = objs.get("filled_flow_data")
station_metadata = objs.get("station_metadata")
station_names = objs.get("station_names")

print("Loaded keys:", list(objs.keys()))
print("Stations in filled_flow_data:", len(filled_flow_data) if filled_flow_data is not None else 0)
print("Stations in station_metadata:", len(station_metadata) if station_metadata is not None else 0)

Loaded keys: ['flow_data', 'flow_data_reindexed', 'filled_flow_data', 'station_metadata', 'station_names']
Stations in filled_flow_data: 32
Stations in station_metadata: 34


In [12]:
### Evaluate a station's Rating Curve and generate metrics and plots ###

def review_rc_originals_only(sid, prefer_dv=True, out_dir=OUT_DIR):
    """
    Main batch evaluation for one station:
    - Evaluates the saved rating curve ("RC") for a station using only "original" observed discharge values.
    - Generates:
        * Predicted vs. observed discharge (with 1:1 line)
        * Predictor (H or Q) vs observed Q, with RC overlay
    - Saves plots and returns fit stats/metadata as dict.
    """
    # Load station metadata, RC string, donor station
    meta = station_metadata.get(sid, {})
    rc_str = (meta.get("Rating Curve (Python Format)") or "").strip()
    donor = (meta.get("Station ID used for rating curve") or "").strip() or sid

    if not rcu.is_valid_rc(rc_str):
        print(f"{sid}: No valid rating curve.")
        return None
    if sid not in filled_flow_data:
        print(f"{sid}: not in filled_flow_data; skipping.")
        return None
    if donor not in flow_data_reindexed:
        print(f"{sid}: donor {donor} not in flow_data_reindexed; skipping.")
        return None

    name = station_names.get(sid, sid)
    kind = rcu.rc_kind(rc_str)  # "H", "Q", "HQ", or "unknown"

    # Load and index dataframes for target and donor
    tgt = rcu.as_idx(filled_flow_data[sid].copy())
    don = rcu.as_idx(flow_data_reindexed[donor].copy())

    # Identify original observations for the target
    tgt_method = tgt.get("Discharge_filled_method", pd.Series(index=tgt.index, dtype=object)).astype(str).str.strip().str.lower()
    tgt_original = (tgt_method.eq("") | tgt_method.eq("original")) & tgt["Discharge"].notna()

    has_stage = "Stage" in don.columns

    # --- Build predictors and mask for evaluation (depends on EVAL_AT setting) ---
    if EVAL_AT == "rc_station":
        don_Q_ok = don["Discharge"].notna() if "Discharge" in don.columns else pd.Series(False, index=don.index)
        mask_idx = don_Q_ok
        y_obs = pd.to_numeric(don.loc[mask_idx, "Discharge"], errors="coerce")
        H_pref = pd.to_numeric(don.get("Stage"), errors="coerce").reindex(y_obs.index) if has_stage else pd.Series(index=y_obs.index, dtype=float)
        Q_in = pd.to_numeric(don.get("Discharge"), errors="coerce").reindex(y_obs.index)
        if has_stage:
            H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
        y_label_overlay = f"Q at {donor}"
        who_sid = donor
    else:
        mask_idx = tgt_original
        y_obs = pd.to_numeric(tgt.loc[mask_idx, "Discharge"], errors="coerce")

        if kind == "Q":
            don_method = don.get("Discharge_filled_method", pd.Series(index=don.index, dtype=object)).astype(str).str.strip().str.lower()
            don_original = (don_method.eq("") | don_method.eq("original")) & don["Discharge"].notna()
            overlap_dates = sorted(set(tgt.loc[mask_idx].index) & set(don.loc[don_original].index))
            y_obs = y_obs.reindex(overlap_dates)
            Q_in = pd.to_numeric(don.loc[overlap_dates, "Discharge"], errors="coerce")
            if has_stage:
                H_pref = pd.to_numeric(don.loc[overlap_dates, "Stage"], errors="coerce")
                H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
            else:
                H_pref = pd.Series(index=y_obs.index, dtype=float)
        else:
            H_pref = pd.to_numeric(don.get("Stage"), errors="coerce").reindex(y_obs.index) if has_stage else pd.Series(index=y_obs.index, dtype=float)
            Q_in = pd.to_numeric(don.get("Discharge"), errors="coerce").reindex(y_obs.index)
            if has_stage:
                H_pref = H_pref.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
            Q_in = Q_in.interpolate(limit=SHORT_GAP_DAYS, limit_direction="both", limit_area="inside")
        y_label_overlay = f"Q at {sid}"
        who_sid = sid

    # Evaluate RC
    H_arr = H_pref.to_numpy() if isinstance(H_pref, pd.Series) else np.zeros(len(y_obs))
    Q_arr = Q_in.to_numpy() if isinstance(Q_in, pd.Series) else np.zeros(len(y_obs))
    y_pred_rc = pd.Series(rcu.eval_rc(rc_str, H_arr, Q_arr), index=y_obs.index)

    # Period masks
    hist = (y_obs.index >= pd.Timestamp(HIST_START)) & (y_obs.index <= pd.Timestamp(HIST_END))
    full = (y_obs.index >= pd.Timestamp(FULL_START)) & (y_obs.index <= pd.Timestamp(FULL_END))
    post = full & (~hist)

    # Stats
    stats_hist_rc = rcu.compute_extended_stats(y_obs[hist].to_numpy(), y_pred_rc[hist].to_numpy())
    stats_full_rc = rcu.compute_extended_stats(y_obs[full].to_numpy(), y_pred_rc[full].to_numpy())
    title_stats = (
        f"HIST: N={stats_hist_rc['N']} r={stats_hist_rc['R']:.2f} R2={stats_hist_rc['R2']:.2f} "
        f"NSE={stats_hist_rc['NSE']:.2f} | "
        f"FULL: N={stats_full_rc['N']} r={stats_full_rc['R']:.2f} R2={stats_full_rc['R2']:.2f} "
        f"NSE={stats_full_rc['NSE']:.2f}"
    )

    # Plot 1: Predicted vs Observed
    finite = np.isfinite(y_obs.to_numpy()) & np.isfinite(y_pred_rc.to_numpy())
    if finite.sum() >= 3:
        both_vals = np.concatenate([y_obs.to_numpy()[finite], y_pred_rc.to_numpy()[finite]])
        lo, hi = rcu.axis_minmax(both_vals, pad_frac=0.03)

        fig1 = plt.figure(figsize=(10, 7.2))
        plt.axline((0, 0), slope=1, color=COLORS["one_to_one"], lw=1, alpha=0.5, label="1:1")
        fmask = pd.Series(finite, index=y_obs.index)
        hist_color, post_color = rcu.obs_colors(PALETTE_MODE, COLORS)
        plt.scatter(y_obs[hist & fmask], y_pred_rc[hist & fmask], s=20, marker="+", c=hist_color, alpha=0.85, label="HIST")
        plt.scatter(y_obs[post & fmask], y_pred_rc[post & fmask], s=20, marker="+", c=post_color, alpha=0.85, label="POST-HIST")
        if lo is not None and hi is not None:
            plt.xlim(lo, hi)
            plt.ylim(lo, hi)
        who_label = f"Q at {who_sid}"
        plt.xlabel(f"{who_label} (Observed)")
        plt.ylabel(f"{who_label} (RC Predicted)")
        who_title = station_names.get(who_sid, who_sid)
        plt.title(f"RC Predicted vs Observed — {who_title}\n{title_stats}")
        plt.legend(loc="upper left")
        plt.tight_layout()
        fn = f"{sid}_{name}_{'RCStation' if EVAL_AT == 'rc_station' else 'Target'}_RC_vs_Obs.png"
        fig1.savefig(out_dir / fn, dpi=PLOT_DPI, bbox_inches="tight")
        plt.close(fig1)

    # Plot 2: Predictor vs Q with RC line
    if kind in ("H", "Q", "HQ"):
        if kind == "H" and has_stage:
            x_series = H_pref
            rc_line_mode = "H"
        elif kind == "Q":
            x_series = Q_in
            rc_line_mode = "Q"
        else:
            if has_stage:
                x_series = H_pref
                rc_line_mode = "H"
            else:
                x_series = Q_in
                rc_line_mode = "Q"

        x_label_overlay = f"H at {donor}" if rc_line_mode == "H" else f"Q at {donor}"

        fmask_pairs = x_series.notna() & y_obs.notna() & y_pred_rc.notna()
        if int(fmask_pairs.sum()) >= 3:
            x_f = x_series[fmask_pairs].to_numpy()
            qobs_f = y_obs[fmask_pairs].to_numpy()
            x_lo, x_hi = rcu.axis_minmax(x_f, pad_frac=0.03)
            y_lo, y_hi = rcu.axis_minmax(qobs_f, pad_frac=0.03)

            fig2, ax = plt.subplots(figsize=(10, 7.2))
            hist_color, post_color = rcu.obs_colors(PALETTE_MODE, COLORS)
            ax.plot(x_series[hist & fmask_pairs], y_obs[hist & fmask_pairs], linestyle="none", marker="+", markersize=4.5, color=hist_color, label="HIST")
            ax.plot(x_series[post & fmask_pairs], y_obs[post & fmask_pairs], linestyle="none", marker="+", markersize=4.5, color=post_color, label="POST-HIST")

            x_grid = np.linspace(np.nanmin(x_f), np.nanmax(x_f), 300)
            if rc_line_mode == "H":
                rc_line = rcu.eval_rc(rc_str, x_grid, np.zeros_like(x_grid))
            else:
                rc_line = rcu.eval_rc(rc_str, np.zeros_like(x_grid), x_grid)

            ax.plot(x_grid, rc_line, color=COLORS["rc_line"], linewidth=2.0, label=f"RC: {rc_str.replace('**', '^')}")
            if x_lo is not None and x_hi is not None:
                ax.set_xlim(x_lo, x_hi)
            if y_lo is not None and y_hi is not None:
                ax.set_ylim(y_lo, y_hi)

            ax.set_xlabel(x_label_overlay)
            ax.set_ylabel(y_label_overlay)
            who_title = station_names.get(sid, sid) if EVAL_AT == "target_station" else station_names.get(donor, donor)
            kind_map = {"H": "Stage–Discharge Rating (H→Q)", "Q": "Index‑gage Rating (Q–Q)", "HQ": "Auxiliary + Index (H & Q)"}
            ax.set_title(f"{kind_map.get(kind, 'Rating')} — {who_title}\n{title_stats}")
            ax.legend(loc="upper left")
            fig2.tight_layout()

            suffix_base = "H_with_RC" if rc_line_mode == "H" else "Q_with_RC"
            suffix = ("RCStation" if EVAL_AT == "rc_station" else "Target") + f"_{suffix_base}.png"
            fig2.savefig(out_dir / f"{sid}_{name}_{suffix}", dpi=PLOT_DPI, bbox_inches="tight")
            plt.close(fig2)

    return {
        "Station ID": sid,
        "Station": name,
        "Donor": donor,
        "RC": rc_str,
        "Kind": kind,
        "eval_at": EVAL_AT,
        "N_hist_RC": stats_hist_rc["N"],
        "R_hist_RC": stats_hist_rc["R"],
        "R2_hist_RC": stats_hist_rc["R2"],
        "RMSE_pct_hist_RC": stats_hist_rc["RMSE_pct"],
        "NSE_hist_RC": stats_hist_rc["NSE"],
        "Bias_pct_hist_RC": stats_hist_rc["Bias_pct"],
        "N_full_RC": stats_full_rc["N"],
        "R_full_RC": stats_full_rc["R"],
        "R2_full_RC": stats_full_rc["R2"],
        "RMSE_pct_full_RC": stats_full_rc["RMSE_pct"],
        "NSE_full_RC": stats_full_rc["NSE"],
        "Bias_pct_full_RC": stats_full_rc["Bias_pct"],
        "prefer_dv": prefer_dv
    }

In [13]:
### Find all RC stations with valid RCs and data ###

rc_station_ids = [
    sid for sid, m in station_metadata.items()
    if rcu.is_valid_rc(m.get("Rating Curve (Python Format)")) and (sid in filled_flow_data)
]

print(f"Found {len(rc_station_ids)} rating curve station(s) with data.")

Found 11 rating curve station(s) with data.


In [14]:
### Evaluate all RCs and collect metrics ###

results, failures = [], []
for sid in rc_station_ids:
    print(f"Evaluating {sid} — {station_names.get(sid, sid)}")
    try:
        res = review_rc_originals_only(sid, prefer_dv=False, out_dir=OUT_DIR)
        if res: results.append(res)
    except Exception as e:
        print(f"  Error: {e}")
        failures.append({"Station ID": sid, "Error": str(e)})

if not results:
    raise RuntimeError("No results produced by evaluation. Double-check that your data and helpers are loaded.")

Evaluating 07381490 — Atchafalaya River at Simmesport, LA
Evaluating 02470629 — Mobile River at River Mile 31 at Bucks, AL
Evaluating 02471019 — Tensaw River near Mount Vernon, AL
Evaluating 07381000 — Bayou Lafourche at Thibodeaux, LA
Evaluating 07381235 — GIWW West of Bayou Lafourche at Larose, LA
Evaluating 07385790 — Charenton Drainage Canal at Baldwin, LA
Evaluating 07386980 — Vermilion River at Perry, LA
Evaluating 08012150 — Mermentau River at Mermentau, LA
Evaluating 08012470 — Bayou Lacassine near Lake Arthur, LA


C:\Users\p00278065\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\p00278065\AppData\Local\anaconda3\Lib\site-packages\numpy\_core\_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Evaluating 08015500 — Calcasieu River near Kinder, LA
Evaluating 08041780 — Neches River at Beaumont, TX


In [16]:
### Post-processing, save summary CSV file ###

# Build DataFrame from batch evaluation results
df = pd.DataFrame(results)

# Add correlation and legacy columns
R_hist_vals, R_full_vals, corr_types, corr_stations = [], [], [], []

for _, row in df.iterrows():
    sid = str(row["Station ID"]).strip()
    donor = str(row.get("Donor", "")).strip() or sid
    rc_str = (station_metadata.get(sid, {}) or {}).get("Rating Curve (Python Format)", "")

    ctype = rcu.infer_corr_type(rc_str)
    corr_types.append(ctype)
    corr_stations.append(donor if ctype in {"Q-H", "Q-Q"} else "")

    if ctype == "--" or sid not in filled_flow_data or donor not in flow_data_reindexed:
        R_hist_vals.append(np.nan)
        R_full_vals.append(np.nan)
        continue

    Q_target = rcu.originals_daily_Q(filled_flow_data[sid], FULL_START, FULL_END)
    donor_series = rcu.donor_daily_series(
        flow_data_reindexed[donor],
        "H" if ctype == "Q-H" else "Q",
        FULL_START,
        FULL_END
    )

    R_hist_vals.append(rcu.pearson_r(Q_target, donor_series, HIST_START, HIST_END))
    R_full_vals.append(rcu.pearson_r(Q_target, donor_series, FULL_START, FULL_END))

df["Correlation Type (from RC)"] = corr_types
df["Correlation Station"] = corr_stations
df["R (HIST)"] = R_hist_vals
df["R (FULL)"] = R_full_vals

# Inline legacy R (2017 CMP)
legacy_csv_text = """Station ID,R (2017 CMP)
03045,1.00
02470629,0.923
02471019,1.00
02479000,0.82
07381000,0.46
07381235,0.63
07381670,0.97
07385790,0.684
07386980,0.49
08012150,0.83
08012470,0.69
08041780,0.90
"""
df_legacy = pd.read_csv(StringIO(legacy_csv_text), dtype={"Station ID": str})
df = df.merge(df_legacy, on="Station ID", how="left")

# Rearrangement for human-friendly CSV
front = [
    "Station ID","Station","Donor","RC","Kind","eval_at",
    "N_hist_RC","R_hist_RC","R2_hist_RC","RMSE_pct_hist_RC","NSE_hist_RC","Bias_pct_hist_RC",
    "N_full_RC","R_full_RC","R2_full_RC","RMSE_pct_full_RC","NSE_full_RC","Bias_pct_full_RC",
    "Correlation Type (from RC)","Correlation Station","R (HIST)","R (FULL)","R (2017 CMP)"
]
other_cols = [c for c in df.columns if c not in front]
cols = [c for c in front if c in df.columns] + other_cols
df_out = df[cols]

# Write to CSV
OUT_CSV = OUT_DIR / "RC_Evaluation_Summary_Combined.csv"
df_out.to_csv(OUT_CSV, index=False)

print("Wrote combined summary CSV:", OUT_CSV)
print(f"Stations evaluated: {len(df_out)}. Failures: {len(failures)}")
if failures:
    print("Failures (first 5):")
    display(pd.DataFrame(failures).head(5))

Wrote combined summary CSV: Plots_Discharge\Evaluate_Rating_Curves\RC_Evaluation_Summary_Combined.csv
Stations evaluated: 11. Failures: 0
